# 01 — O que são Embeddings?

Antes de qualquer código, vamos entender o problema que os embeddings resolvem.

Modelos de machine learning só trabalham com números. Mas texto é linguagem humana — cheio de contexto, ambiguidade e significado. Como transformar uma frase em números de forma que o *significado* seja preservado?

A resposta ingênua seria: contar palavras (bag-of-words). Problema: "banco de dados" e "sentar no banco" usam a mesma palavra, mas significam coisas completamente diferentes.

A resposta moderna é o **embedding**: um vetor de centenas de números aprendido por uma rede neural treinada em bilhões de frases. A rede aprendeu a mapear o *significado* — não só as palavras — para posições num espaço geométrico.

**O que você vai aprender neste notebook:**
- O que é um embedding e como ele funciona intuitivamente
- Como medir similaridade entre textos usando álgebra linear simples
- Como visualizar significado em 2D e 3D
- Por que podemos fazer "aritmética de significados" com vetores

## 1.1 O problema: computadores não entendem palavras

Modelos de machine learning trabalham com **números**. Mas texto é... texto.

Como representar:
- `"O gato dorme"` como números?
- De forma que `"O felino repousa"` seja **próximo** a `"O gato dorme"`?
- E `"A bolsa caiu"` seja **distante** de ambos?

Resposta: **Embeddings** — vetores densos aprendidos por modelos de linguagem.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# Nossas frases de exemplo
frases = [
    "O gato dorme no sofá",           # 0
    "O felino repousa no divã",        # 1 - semanticamente similar a 0
    "Um cachorro late para o carteiro",# 2 - animal, mas diferente
    "A bolsa de valores caiu hoje",    # 3 - completamente diferente
    "Machine learning usa redes neurais",  # 4
    "Deep learning é um tipo de IA",   # 5 - similar a 4
    "Python é uma linguagem de programação",  # 6
    "Receita de bolo de chocolate",    # 7 - totalmente diferente de 4-6
]

# Criar embeddings (cada frase vira um vetor de 384 números)
embeddings = model.encode(frases, normalize_embeddings=True)

print(f"Shape: {embeddings.shape}")
print(f"→ {len(frases)} frases, cada uma representada por {embeddings.shape[1]} números")
print(f"\nPrimeiros 10 valores do embedding da frase 0:")
print(f"  {embeddings[0, :10].round(4)}")

## 1.2 Intuição Geométrica

Agora que geramos os embeddings, vamos entender o que esses números *significam* geometricamente.

Imagine cada embedding como um ponto no espaço — mas em vez de 3 dimensões (x, y, z), temos 384.

A **similaridade de cosseno** mede o ângulo entre dois vetores:
- **1.0** = vetores idênticos (mesma direção)
- **0.0** = vetores ortogonais (sem relação)
- **-1.0** = vetores opostos (raro em embeddings de texto)

A intuição: textos com significado parecido apontam para a *mesma direção* no espaço de 384 dimensões. A magnitude (comprimento do vetor) não importa — só a direção.

> **Por que cosseno e não distância euclidiana?** Porque a distância euclidiana é afetada pelo comprimento do vetor. Frases mais longas geram vetores com norma maior, o que distorceria a comparação. O cosseno elimina esse efeito.

A célula abaixo calcula todas as similaridades par-a-par entre nossas 8 frases:

In [ ]:
import pandas as pd

# Calcular matriz de similaridade cosine
# (dot product de vetores normalizados = cosine similarity)
sim_matrix = embeddings @ embeddings.T

labels_curtos = [f[:25] + "..." if len(f) > 25 else f for f in frases]
df_sim = pd.DataFrame(sim_matrix.round(3), index=labels_curtos, columns=labels_curtos)

print("Matriz de Similaridade Cosine (1.0 = idêntico, 0.0 = ortogonal):\n")
print(df_sim.to_string())

print("\n🔍 Observações:")
print(f"  'gato dorme' × 'felino repousa':    {sim_matrix[0,1]:.3f}  ← Alta similaridade!")
print(f"  'gato dorme' × 'bolsa caiu':        {sim_matrix[0,3]:.3f}  ← Baixa similaridade")
print(f"  'machine learning' × 'deep learning': {sim_matrix[4,5]:.3f}  ← Alta similaridade!")
print(f"  'machine learning' × 'bolo de chocolate': {sim_matrix[4,7]:.3f}  ← Baixa")

### O que os números nos dizem?

Leia a tabela de cima para baixo e observe:

- **"gato dorme" × "felino repousa" = ~0.58** → alta similaridade, mesmo sem nenhuma palavra em comum. O modelo entendeu que "gato" e "felino" são a mesma coisa, e "dorme" e "repousa" também.
- **"machine learning" × "deep learning" = ~0.42** → ambos são conceitos de IA, mas o modelo distingue que não são idênticos.
- **"machine learning" × "bolo de chocolate" = ~0.10** → praticamente nenhuma relação semântica.

**Insight chave:** a similaridade de cosseno captura *relacionamento semântico*, não *sobreposição de palavras*. Isso é o que torna embeddings úteis para busca — você pode encontrar documentos relevantes mesmo que eles usem vocabulário diferente da sua query.

## 1.3 Visualização 2D com PCA

384 dimensões é impossível de visualizar diretamente. A técnica **PCA (Principal Component Analysis)** comprime essas dimensões para 2, preservando o máximo de variância possível.

Pense assim: se você tivesse uma nuvem de pontos em 3D e precisasse fotografá-la de um ângulo só, você escolheria o ângulo que mostra mais estrutura. O PCA faz isso matematicamente, escolhendo as 2 "direções" que mais separam os dados.

**O que esperar:** frases sobre animais deverão aparecer próximas umas das outras, frases sobre tecnologia em outro cluster, e assim por diante. Se o modelo tiver aprendido bem, os clusters visuais vão fazer sentido semântico.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Cores por categoria
cores = ["#e74c3c", "#e74c3c",  # animais (vermelho)
         "#e67e22",              # animal diferente
         "#3498db",             # finanças (azul)
         "#2ecc71", "#2ecc71", "#27ae60",  # tecnologia (verde)
         "#9b59b6"]             # culinária (roxo)

categorias = ["Animais", "Animais", "Animais", "Finanças",
              "Tecnologia", "Tecnologia", "Tecnologia", "Culinária"]

fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
           c=cores, s=200, zorder=3, edgecolors='white', linewidths=2)

for i, (x, y) in enumerate(embeddings_2d):
    ax.annotate(
        frases[i],
        (x, y),
        textcoords="offset points",
        xytext=(8, 5),
        fontsize=9,
        wrap=True,
    )

patches = [
    mpatches.Patch(color='#e74c3c', label='Animais'),
    mpatches.Patch(color='#3498db', label='Finanças'),
    mpatches.Patch(color='#2ecc71', label='Tecnologia'),
    mpatches.Patch(color='#9b59b6', label='Culinária'),
]
ax.legend(handles=patches, loc='upper left', fontsize=10)

ax.set_title("Embeddings projetados em 2D (PCA)\nFrases similares ficam próximas!", 
             fontsize=14, fontweight='bold')
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("pca_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("💡 Observe como frases da mesma categoria ficam agrupadas!")

### Conclusão: geometria = semântica

O gráfico confirma o que a tabela de similaridade sugeriu: o modelo MiniLM organizou o espaço vetorial de forma que **categorias semânticas formam clusters visíveis**.

Isso não foi programado explicitamente — emergiu do treinamento em bilhões de pares de frases. O modelo aprendeu que "gato" e "felino" devem ficar próximos, e que ambos devem ficar longe de "bolsa de valores".

**Por que isso importa para RAG?** Quando você busca "qual o preço das ações hoje?", o sistema encontra documentos sobre finanças — mesmo que usem palavras como "cotação", "mercado", "índice" — porque todos esses conceitos ocupam a mesma região do espaço vetorial.

## 1.4 Visualização 3D interativa com Plotly

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA

pca3 = PCA(n_components=3)
emb_3d = pca3.fit_transform(embeddings)

df_3d = pd.DataFrame({
    "PC1": emb_3d[:, 0],
    "PC2": emb_3d[:, 1],
    "PC3": emb_3d[:, 2],
    "texto": frases,
    "categoria": categorias,
})

fig = px.scatter_3d(
    df_3d, x='PC1', y='PC2', z='PC3',
    color='categoria',
    text='texto',
    title='Embeddings em 3D (PCA) — Rotacione para ver os clusters!',
    width=900, height=600,
)

fig.update_traces(textposition='top center', textfont_size=9, marker_size=8)
from IPython.display import display, HTML
import plotly.io as pio
display(HTML(pio.to_html(fig, full_html=False, include_plotlyjs="cdn")))

## 1.5 Analogias Vetoriais: Álgebra de Significado

Uma das descobertas mais surpreendentes sobre embeddings: você pode fazer **aritmética com significados**.

O exemplo clássico:

```
vetor("rei") - vetor("homem") + vetor("mulher") ≈ vetor("rainha")
```

Isso funciona porque o modelo aprendeu que a relação "rei→rainha" é similar à relação "ator→atriz" ou "professor→professora" — uma transformação de gênero. Essa transformação existe como uma *direção* consistente no espaço vetorial.

**Cuidado com o experimento abaixo:** usamos um vocabulário de apenas 12 palavras. Analogias vetoriais funcionam bem com vocabulários grandes (WordVec tem 3 milhões de palavras). Com 12 palavras, o "vizinho mais próximo" é forçado a ser uma das 12 opções — e o resultado pode ser aleatório.

In [ ]:
words = ["rei", "rainha", "homem", "mulher", "Paris", "França", "Berlim", "Alemanha",
         "Python", "programação", "Java", "computador"]

word_embeddings = model.encode(words, normalize_embeddings=True)
word_dict = {w: emb for w, emb in zip(words, word_embeddings)}

def analogia(a, b, c, vocab=word_dict, top_k=3):
    """a - b + c = ?"""
    resultado = word_dict[a] - word_dict[b] + word_dict[c]
    resultado = resultado / np.linalg.norm(resultado)
    
    candidatos = [(w, float(resultado @ emb)) 
                  for w, emb in word_dict.items() 
                  if w not in (a, b, c)]
    candidatos.sort(key=lambda x: -x[1])
    
    return candidatos[:top_k]

print("Analogias vetoriais com embeddings:\n")
print(f"rei - homem + mulher = ?")
print(f"  Top: {analogia('rei', 'homem', 'mulher')}")

print(f"\nParis - França + Alemanha = ?")
print(f"  Top: {analogia('Paris', 'França', 'Alemanha')}")

print(f"\nPython - programação + computador = ?")
print(f"  Top: {analogia('Python', 'programação', 'computador')}")

print("\n💡 Nota: com vocabulário pequeno, os resultados são limitados.")
print("   Com modelos maiores e vocabulário maior, as analogias ficam muito mais precisas!")

### Por que os resultados parecem estranhos?

Com apenas 12 palavras no vocabulário, o modelo não tem opções. Se o resultado correto de "rei - homem + mulher" fosse "rainha" mas o vetor calculado ficasse mais próximo de "Paris" por acaso, é isso que aparece.

**Em modelos reais** (Word2Vec com 3M palavras, GloVe, FastText), as analogias funcionam surpreendentemente bem:
- rei - homem + mulher = rainha ✓
- Paris - França + Itália = Roma ✓
- melhor - bom + ruim = pior ✓

O fenômeno existe e é real — só precisamos de vocabulário suficiente para demonstrá-lo.

**Insight profundo:** isso prova que os embeddings não são apenas "lookup de palavras". Eles codificam *relações* entre conceitos como direções geométricas consistentes. Gênero, país-capital, grau de comparação — tudo são direções no espaço vetorial.

## 1.6 Como os Embeddings são Criados?

Entender o processo de criação ajuda a saber as limitações e quando os embeddings vão (ou não vão) funcionar bem.

O modelo `all-MiniLM-L6-v2` usa uma arquitetura **BERT-like** com 6 camadas Transformer:

```
Texto → Tokenização → 6x Transformer → Pooling → Projeção → Vetor 384d
```

**Como ele foi treinado?** Com *contrastive learning* em ~1 bilhão de pares de frases:
- Pares positivos: "O gato dorme" + "O felino repousa" → vetores devem ser próximos
- Pares negativos: "O gato dorme" + "A bolsa caiu" → vetores devem ser distantes

A rede ajusta seus pesos para satisfazer essas restrições em bilhões de exemplos. O resultado emergente: um espaço onde proximidade geométrica = similaridade semântica.

**Limitações importantes:**
- O modelo não "entende" português — ele aprendeu padrões estatísticos
- Funciona melhor para frases no domínio do seu treinamento
- Frases muito longas (>256 tokens) são truncadas

In [ ]:
# Ver informações do modelo
print(f"Modelo: {model._model_card_text[:200] if hasattr(model, '_model_card_text') else 'all-MiniLM-L6-v2'}")
print(f"\nDimensões de saída: {model.get_sentence_embedding_dimension()}")
print(f"Máximo de tokens: {model.max_seq_length}")

# Ver quantos parâmetros o modelo tem
total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros: {total_params:,} (~{total_params/1e6:.0f}M)")

### Tamanho importa?

O MiniLM tem 22 milhões de parâmetros e gera vetores de 384 dimensões. Por comparação:
- **MPNet (768d):** 110M parâmetros — 5x maior, melhor qualidade
- **GPT-3.5 (embedding):** bilhões de parâmetros — mas você paga por API call
- **nomic-embed (768d):** roda local via Ollama, qualidade comparável ao MPNet

**Regra prática:** MiniLM para prototipar (rápido, leve). MPNet para produção (melhor qualidade). Modelos via API quando você não quer gerenciar infraestrutura.

---

## Resumo do Notebook

| Conceito | O que aprendemos |
|---------|------------------|
| **Embedding** | Vetor de números que representa o *significado* de um texto |
| **Dimensão** | Cada número no vetor — 384 "slots" para codificar significado |
| **Similaridade cosine** | Mede ângulo entre vetores — ignora magnitude, captura direção |
| **Clusters semânticos** | Textos sobre o mesmo assunto ocupam a mesma região do espaço |
| **Aritmética vetorial** | Operações geométricas correspondem a operações de significado |
| **Treinamento** | Contrastive learning em 1B pares — emergência, não programação explícita |

**O que vem a seguir:** agora que você entende o que é um embedding, o próximo notebook explora por que diferentes modelos usam 384, 768 ou 1024 dimensões — e o impacto real em memória, velocidade e qualidade.

- [02 — Dimensões Vetoriais](02_vector_dimensions.html): por que 384 vs 768 vs 1024?
- [03 — Float Types](03_float_types.html): como comprimir embeddings sem perder qualidade
- [04 — Métricas de Distância](04_distance_metrics.html): quando usar cosine vs euclidean

## Resumo

| Conceito | O que aprendemos |
|---------|------------------|
| Embedding | Vetor de números que representa o significado de um texto |
| Dimensão | Tamanho do vetor (MiniLM = 384, MPNet = 768) |
| Similaridade cosine | Mede o ângulo entre dois vetores (0 = idênticos) |
| PCA/UMAP | Reduz dimensionalidade para visualização |
| Analogias | Podemos fazer aritmética com significados! |

## Próximos passos

- [02 — Vector Dimensions](02_vector_dimensions.html): Por que 384 vs 768 vs 1024?
- [03 — Float Types](03_float_types.html): Como comprimir embeddings sem perder qualidade?
- [04 — Distance Metrics](04_distance_metrics.html): Quando usar cosine vs euclidean?